# Netflix Movies and TV Shows: Explainable AI Learning Workflow

This notebook is the primary step-by-step learning entry point for this dataset project. Run the cells from top to bottom. The notebook contains the complete analysis logic, so it can be sent together with the dataset folder without any shared source-code directory.

The optional `app.py` remains available for the interactive Streamlit dashboard; this notebook is the recommended starting point for classroom practice.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = Path.cwd()
DATA_PATH = ROOT / "netflix_titles.csv"


@dataclass
class NetflixRun:
    data: pd.DataFrame
    vectorizer: TfidfVectorizer
    matrix: object



## Step 1: Define the explainable recommendation pipeline

TF-IDF represents metadata and description text. Cosine similarity ranks candidates, and shared high-weight terms explain each recommendation.


In [ ]:

def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH).drop_duplicates(subset=["show_id"]).copy()
    fields = df[["type", "listed_in", "description", "country", "director", "cast"]].fillna("").astype(str)
    df["content_text"] = fields.agg(" ".join, axis=1)
    return df.reset_index(drop=True)


def train() -> NetflixRun:
    data = load_data()
    vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=25000, ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(data["content_text"])
    return NetflixRun(data, vectorizer, matrix)


def recommend(run: NetflixRun, title: str, n: int = 8) -> pd.DataFrame:
    index = int(run.data.index[run.data["title"] == title][0])
    query = run.matrix[index]
    scores = (run.matrix @ query.T).toarray().ravel()
    candidates = np.argsort(scores)[::-1]
    rows: list[dict[str, object]] = []
    terms = np.asarray(run.vectorizer.get_feature_names_out())
    for candidate in candidates:
        if candidate == index:
            continue
        shared = query.multiply(run.matrix[candidate]).toarray().ravel()
        term_ids = np.argsort(shared)[::-1]
        shared_terms = [terms[i] for i in term_ids if shared[i] > 0][:4]
        rows.append({"title": run.data.iloc[candidate]["title"], "type": run.data.iloc[candidate]["type"], "genres": run.data.iloc[candidate]["listed_in"], "similarity": float(scores[candidate]), "shared_explanation_terms": ", ".join(shared_terms)})
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)


def coverage_summary(run: NetflixRun) -> pd.DataFrame:
    return run.data.groupby("type").agg(titles=("show_id", "size"), median_release_year=("release_year", "median")).reset_index()


## Step 2: Build the content index


In [ ]:
run = train()
print(f'Titles: {len(run.data):,}')
run.data[['title', 'type', 'listed_in', 'release_year']].head()


## Step 3: Retrieve recommendations with explanation terms


In [ ]:
selected_title = run.data.iloc[0]['title']
print('Selected title:', selected_title)
recommend(run, selected_title, n=8)


## Step 4: Check catalogue coverage


In [ ]:
coverage_summary(run)
